### Skill Extraction
Use the stricter extractor filters so output focuses on canonical, meaningful skills (not years, stopwords, or boilerplate terms).

In [1]:
import sys
from pathlib import Path

# add ../src to Python path (robust for notebook location)
src_path = (Path.cwd().parent / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


import importlib
import skills
import skills_extraction

importlib.reload(skills)
importlib.reload(skills_extraction)

from skills_extraction import SkillExtractor, compute_extraction_stats
from skills import DEFAULT_SYNONYM_MAP

import json
import pandas as pd

In [2]:
processed_dir = "../data/processed"

# resumes_df = pd.read_csv(f"{processed_dir}/resumes_cleaned.csv")
# jobs_df = pd.read_csv(f"{processed_dir}/jobs_merged_cleaned.csv")
resumes_df = pd.read_csv(f"{processed_dir}/resumes_subset.csv")
jobs_df = pd.read_csv(f"{processed_dir}/jobs_subset.csv")

print("resumes:", resumes_df.shape)
print("jobs:", jobs_df.shape)

resumes: (712, 4)
jobs: (3000, 15)


In [3]:
extractor = SkillExtractor.from_vocabulary_json(
    vocab_path=f"{processed_dir}/skill_vocabulary.json",
    synonym_map=DEFAULT_SYNONYM_MAP,
    min_alias_len=2,
    min_vocab_frequency=10  # ignore ultra-rare/noisy vocabulary entries
)
print("Extractor ready.")
print("Usable canonical vocabulary size:", len(extractor.skill_vocab))

Extractor ready.
Usable canonical vocabulary size: 141966


In [4]:
# Resume skill extraction
resume_out = extractor.extract_from_dataframe(
    resumes_df,
    text_col="Resume_cleaned",
    output_col="extracted_skills",
    max_skills_per_text=300
)

resume_stats = compute_extraction_stats(resume_out, skills_col="extracted_skills", top_n=20)
print(resume_stats)

# Job skill extraction
job_out = extractor.extract_from_dataframe(
    jobs_df,
    text_col="job_skills_cleaned",
    output_col="extracted_skills",
    max_skills_per_text=300
)

job_stats = compute_extraction_stats(job_out, skills_col="extracted_skills", top_n=20)
print(job_stats)


print("resume rows:", len(resume_out), "job rows:", len(job_out))
print("avg resume skills:", round(resume_out["extracted_skill_count"].mean(), 2))
print("avg job skills:", round(job_out["extracted_skill_count"].mean(), 2))


{'total_rows': 712, 'rows_with_skills': 712, 'coverage_pct': 100.0, 'avg_skills_per_row': 234.94, 'top_skills': [('team', 513), ('training', 464), ('customer', 432), ('business', 431), ('university', 411), ('information', 406), ('manager', 394), ('other', 372), ('staff', 356), ('communication', 350), ('sales', 339), ('project', 335), ('high', 321), ('ensure', 311), ('responsible', 309), ('data', 307), ('planning', 307), ('quality', 301), ('daily', 300), ('microsoft', 296)]}
{'total_rows': 3000, 'rows_with_skills': 2858, 'coverage_pct': 95.27, 'avg_skills_per_row': 44.52, 'top_skills': [('communication', 1556), ('customer', 750), ('problem solving', 732), ('customer service', 631), ('leadership', 621), ('license', 552), ('team', 536), ('communication skills', 535), ('teamwork', 531), ('degree', 529), ('safety', 501), ('training', 483), ('microsoft', 477), ('certification', 447), ('planning', 440), ('health', 437), ('analysis', 421), ('data', 420), ('quality', 386), ('project', 382)]}
re

In [5]:
# Quick manual sanity check
sample_cols = ["ID", "Category", "extracted_skills", "extracted_skill_count"]
display(resume_out[sample_cols].sample(5, random_state=42))
print(resume_out.loc[0, "extracted_skills"])

sample_cols = ["extracted_skills", "extracted_skill_count"]
display(job_out[sample_cols].sample(5, random_state=42))
print(job_out.loc[0, "extracted_skills"])

,ID,Category,extracted_skills,extracted_skill_count
506,20457611,FITNESS,"[account executives, account manager, accounti...",300
394,37058472,DESIGNER,"[3d rendering, accurate, action, action plans,...",207
210,26673507,BANKING,"[accounting, accuracy, achievement, achievemen...",245
247,69097572,BPO,"[accenture, account, account managers, account...",273
437,11005406,DIGITAL-MEDIA,"[administration, administration systems, amazo...",204


['account', 'account reconciliation', 'accountant', 'accounting', 'accounting department', 'accounting duties', 'accounting manager', 'accounting services', 'accounting software', 'accounts', 'accounts payable', 'accounts receivable', 'agency', 'analysis', 'annual', 'arizona', 'as needed', 'assist', 'audits', 'bank', 'bank reconciliations', 'billing', 'budget', 'budget analysis', 'business', 'business development', 'business management', 'businesses', 'charge', 'clients', 'cobra', 'coffee', 'coffee shop', 'commercial', 'commercial leasing', 'companies', 'company policies', 'construction', 'construction contractors', 'consulting', 'consulting services', 'contractors', 'corporations', 'customer', 'customer service', 'deadlines', 'department', 'directors', 'donations', 'duties', 'effective', 'effective time management', 'employee', 'employee management', 'employee paperwork', 'employee performance', 'employee performance evaluations', 'employee policies', 'employees', 'engineering', 'enti

,extracted_skills,extracted_skill_count
1801,"[cleanliness, communication, customer, custome...",39
1190,"[applications, computer, computer applications...",16
1817,"[business, business development, carriers, col...",31
251,"[action, adjunct, adjunct professor, affirmati...",33
2505,"[accounting, accounting and finance, certifica...",42


['charting', 'electronic charting', 'flexible', 'flexible scheduling', 'health', 'nursing', 'nursing care', 'paid', 'paid training', 'pediatrics', 'plan', 'referral', 'savings', 'scheduling', 'training']


In [6]:
# Use spacy and pos gate to manually check extracted skills for sanity
import re
import math
from collections import Counter
import spacy

# Load once
nlp = spacy.load("en_core_web_sm")

def build_idf(skill_lists):
    n_docs = len(skill_lists)
    df = Counter()
    for skills in skill_lists:
        if isinstance(skills, list):
            df.update(set(skills))
    return {k: math.log((n_docs + 1) / (v + 1)) + 1.0 for k, v in df.items()}

def keep_by_pos(term: str, vocab=None) -> bool:
    term = (term or "").strip().lower()
    if not term:
        return False

    if not re.search(r"[a-z0-9+#./-]", term):
        return False

    # allow if known skill (bypass spaCy)
    if vocab and term in vocab:
        return True

    doc = nlp(term)

    if len(doc) == 1:
        t = doc[0]
        if t.is_stop:
            return False
        if t.pos_ not in {"NOUN", "PROPN"}:
            return False
    else:
        if not any(tok.pos_ in {"NOUN", "PROPN"} for tok in doc):
            return False

    return True

def family_key(skill: str) -> str:
    s = skill.lower().strip()
    first = s.split()[0] if s else s
    first = re.sub(r"(ing|ed|s|ion|ions|al|ive|er|or)$", "", first)
    return first

def filter_and_rerank(skills, idf_map, top_k=None, min_idf=1.15, max_per_family=2):
    if not isinstance(skills, list):
        return []

    unique_skills = sorted(set(s for s in skills if isinstance(s, str) and s.strip()))

    # 1) POS/shape filter
    filtered = [s for s in unique_skills if keep_by_pos(s, vocab=idf_map.keys())]

    # 2) Specificity filter
    # Keep all multi-word phrases, and only sufficiently specific single-word terms
    filtered = [
        s for s in filtered
        if (len(s.split()) >= 2) or (idf_map.get(s, 0.0) >= min_idf)
    ]

    # 3) Score + diversity
    ranked = sorted(
        filtered,
        key=lambda s: (idf_map.get(s, 0.0) + 0.08 * min(len(s.split()), 4)),
        reverse=True
    )

    selected = []
    fam_count = {}
    for s in ranked:
        fam = family_key(s)
        if fam_count.get(fam, 0) >= max_per_family:
            continue
        selected.append(s)
        fam_count[fam] = fam_count.get(fam, 0) + 1
        if top_k is not None and len(selected) >= top_k:
            break

    return selected

# Build corpus IDF from both resume + job extractions
all_skill_lists = resume_out["extracted_skills"].tolist() + job_out["extracted_skills"].tolist()
idf_map = build_idf(all_skill_lists)

# top_k=None means no hard cap; keep all filtered/ranked skills
resume_out["extracted_skills_ranked"] = resume_out["extracted_skills"].apply(
    lambda x: filter_and_rerank(x, idf_map, top_k=None, min_idf=1.15, max_per_family=2)
)
job_out["extracted_skills_ranked"] = job_out["extracted_skills"].apply(
    lambda x: filter_and_rerank(x, idf_map, top_k=None, min_idf=1.15, max_per_family=2)
)

resume_out["extracted_skill_count_ranked"] = resume_out["extracted_skills_ranked"].apply(len)
job_out["extracted_skill_count_ranked"] = job_out["extracted_skills_ranked"].apply(len)

print("resume avg ranked:", round(resume_out["extracted_skill_count_ranked"].mean(), 2))
print("job avg ranked:", round(job_out["extracted_skill_count_ranked"].mean(), 2))
print("resume sample:", resume_out.loc[0, "extracted_skills_ranked"][:40])

resume avg ranked: 212.46
job avg ranked: 41.15
resume sample: ['general ledger review', 'industry specific software', 'international business development', 'accounting services', 'coffee shop', 'commercial leasing', 'construction contractors', 'employee performance evaluations', 'sales tax reporting', 'multiple entities', 'player experience', 'service education', 'accounting duties', 'employee paperwork', 'fund accounting', 'monthly billing', 'payroll functions', 'payroll tax', 'procedure manual', 'tax reporting', 'quicken', 'use tax', 'paychex', 'subcontracting', 'quickbooks pro', 'cobra', 'weekly payroll', 'effective time management', 'general ledger accounting', 'managing multiple projects', 'coffee', 'monthly financial statements', 'donations', 'meeting deadlines', 'staff accountant', 'financial statement preparation', 'sales revenue', 'financial statement analysis', 'consulting services', 'statement preparation']


In [7]:
# Quick manual sanity check
sample_cols = ["ID", "Category", "extracted_skills_ranked", "extracted_skill_count_ranked"]
display(resume_out[sample_cols].sample(5, random_state=42))
print(resume_out.loc[0, "extracted_skills_ranked"])

sample_cols = ["extracted_skills_ranked", "extracted_skill_count_ranked"]
display(job_out[sample_cols].sample(5, random_state=42))
print(job_out.loc[0, "extracted_skills_ranked"])

,ID,Category,extracted_skills_ranked,extracted_skill_count_ranked
506,20457611,FITNESS,"[written and spoken communication, written and...",264
394,37058472,DESIGNER,"[assembly techniques, conceptual design, exhib...",185
210,26673507,BANKING,"[banking solutions, credit requests, external ...",221
247,69097572,BPO,"[basis of estimate, sap business warehouse, ar...",263
437,11005406,DIGITAL-MEDIA,"[cisco asa firewall, administration systems, b...",181


['general ledger review', 'industry specific software', 'international business development', 'accounting services', 'coffee shop', 'commercial leasing', 'construction contractors', 'employee performance evaluations', 'sales tax reporting', 'multiple entities', 'player experience', 'service education', 'accounting duties', 'employee paperwork', 'fund accounting', 'monthly billing', 'payroll functions', 'payroll tax', 'procedure manual', 'tax reporting', 'quicken', 'use tax', 'paychex', 'subcontracting', 'quickbooks pro', 'cobra', 'weekly payroll', 'effective time management', 'general ledger accounting', 'managing multiple projects', 'coffee', 'monthly financial statements', 'donations', 'meeting deadlines', 'staff accountant', 'financial statement preparation', 'sales revenue', 'financial statement analysis', 'consulting services', 'statement preparation', 'budget analysis', 'corporations', 'receivables', 'tax preparation', 'arizona', 'entities', 'experts', 'international business', '

,extracted_skills_ranked,extracted_skill_count_ranked
1801,"[human resources operations, markdown executio...",38
1190,"[pharmacy technician license, pharmacy technic...",14
1817,"[thirdparty logistics, sales negotiation, thir...",29
251,"[letter of application, persons with disabilit...",32
2505,"[loud conditions, nonslip shoes, college class...",41


['electronic charting', 'flexible scheduling', 'paid training', 'charting', 'pediatrics', 'nursing care', 'referral', 'savings', 'paid', 'flexible', 'scheduling', 'nursing', 'plan', 'health', 'training']


In [8]:
# Save ranked/filtered outputs (store list as JSON string for stable CSV format)
resume_save = resume_out.copy()
resume_save["extracted_skills"] = resume_save["extracted_skills"].apply(json.dumps)
resume_save["extracted_skills_ranked"] = resume_save["extracted_skills_ranked"].apply(json.dumps)
resume_save.to_csv(f"{processed_dir}/resume_skills_extracted.csv", index=False)

job_save = job_out.copy()
job_save["extracted_skills"] = job_save["extracted_skills"].apply(json.dumps)
job_save["extracted_skills_ranked"] = job_save["extracted_skills_ranked"].apply(json.dumps)
job_save.to_csv(f"{processed_dir}/job_skills_extracted.csv", index=False)

print("Saved:")
print(f"{processed_dir}/resume_skills_extracted.csv")
print(f"{processed_dir}/job_skills_extracted.csv")

Saved:
../data/processed/resume_skills_extracted.csv
../data/processed/job_skills_extracted.csv
